# E2a — Perovskite-specialized ChargeFlow (Colab, resumable)

**Goal:** chemistry-homogeneous proof-of-principle — `<1%` ε_MAE on a parent-disjoint perovskite split.

**Runtime:** Runtime → Change runtime type → **T4 GPU (free tier)**. Do NOT pay for A100/L4.

> Each perovskite has a different 3D grid size, so training uses `batch_size=1`
> with `accum_iter=16` for an effective batch of 16. The model is tiny
> (`unet_cond`, model_channels=4) — well under T4's 16 GB even at native resolution.

**Estimated cost:** $0 — free Colab T4 is sufficient for all 300 epochs.

## 0. Verify GPU

In [1]:
!nvidia-smi


Tue Jun 23 00:13:49 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Mount Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/chargeflow'
DATA_DIR   = f'{DRIVE_ROOT}/data_perovskite'
OUTPUT_DIR = f'{DRIVE_ROOT}/output_perovskite'
LISTS_DIR  = f'{DRIVE_ROOT}/lists_perovskite'
for d in (DRIVE_ROOT, DATA_DIR, OUTPUT_DIR, LISTS_DIR):
    os.makedirs(d, exist_ok=True)
print('Drive ready:', DRIVE_ROOT)


Mounted at /content/drive
Drive ready: /content/drive/MyDrive/chargeflow


## 2. Clone repo

In [3]:
import os, shutil
REPO_DIR = '/content/chargeflow'

%cd /content

# Remove stale clone if present
if os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)
    print('Removed old clone.')

!git clone https://github.com/ngminhtri0394/chargeflow-electron-density-main.git {REPO_DIR}

# The GitHub repo may have a nested subfolder — find where scripts/ actually lives
actual_root = REPO_DIR
for entry in os.listdir(REPO_DIR):
    candidate = f'{REPO_DIR}/{entry}'
    if os.path.isdir(f'{candidate}/scripts'):
        actual_root = candidate
        print(f'Found scripts/ inside subfolder: {entry}')
        break
REPO_DIR = actual_root

%cd {REPO_DIR}
assert os.path.isdir(f'{REPO_DIR}/scripts'), f'scripts/ not found under {REPO_DIR}, contents: {os.listdir(REPO_DIR)}'
print('Done. Repo ready at', REPO_DIR)


/content
Cloning into '/content/chargeflow'...
remote: Enumerating objects: 270, done.
remote: Counting objects: 100% (270/270), done.
remote: Compressing objects: 100% (159/159), done.
remote: Total 270 (delta 146), reused 233 (delta 109), pack-reused 0 (from 0)
Receiving objects: 100% (270/270), 828.26 KiB | 17.62 MiB/s, done.
Resolving deltas: 100% (146/146), done.
Found scripts/ inside subfolder: chargeflow-electron-density-main
/content/chargeflow/chargeflow-electron-density-main
Done. Repo ready at /content/chargeflow/chargeflow-electron-density-main


In [4]:
# Install all deps: use pip install -e . first (picks up setup.py),
# then add packages not listed in requirements.txt that src/ actually imports.
!pip -q install -e .
!pip -q install flow_matching ase pymatgen torchmetrics torchvision pyyaml
import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())


  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 72.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.9/57.9 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.7/89.7 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 95.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.8/269.8 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.1/121.1 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.6/85.6 kB 9.9 MB/s eta 0:00:00
   

## 3. Write config (embedded — no upload needed)

The config lives directly in this cell so it works even on a fresh clone that doesn't have `config/train_perovskite_specialized.yaml`.

In [5]:
import yaml, os

cfg = {
    'model': {
        'architecture': 'unet_cond_xlarge',   # C=16; resolves near-core detail
        'discrete_flow_matching': False,
        'use_ema': True,
    },
    'data': {
        'train_data_list':      f'{LISTS_DIR}/list_d_perovskite_train',
        'train_label_list':     f'{LISTS_DIR}/list_l_perovskite_train',
        'train_data_gridsize':  f'{LISTS_DIR}/list_dgs_perovskite_train',
        'train_label_gridsize': f'{LISTS_DIR}/list_lgs_perovskite_train',
        'use_charge_dataset':   True,
        'data_augmentation':    True,
        'downsample_data':      1,
        'downsample_label':     1,
        'normalize_density':    False,
        'batch_size':           1,     # variable 3D grids
        'num_workers':          2,
        'pin_memory':           True,
    },
    'training': {
        'epochs':           2000,
        'start_epoch':      0,
        'optimizer':        'AdamW',
        'learning_rate':    1e-4,
        'optimizer_betas':  [0.9, 0.95],
        'decay_lr':         True,
        'accum_iter':       16,
        'class_drop_prob':  0.1,
        # *** KEY FIX: paper's formulation is SAD -> DFT flow, not noise -> residual. ***
        # start_sad=True makes the flow start from the SAD density (x_0=SAD, x_1=DFT).
        'start_sad':        True,
        'core_weight':      1.0,       # near-core-weighted loss (plan R1.3)
        'save_frequency':   10,
        'save_best':        True,
        'log_frequency':    10,
    },
    'evaluation': {'eval_frequency': 50, 'compute_fid': False, 'save_samples': True},
    'ode': {
        'method': 'dopri5', 'options': {'atol': 1e-5, 'rtol': 1e-5},
        'cfg_scale': 1.0, 'sampling_dtype': 'float32',
        'skewed_timesteps': True, 'edm_schedule': True,
    },
    'distributed': {'enabled': False, 'backend': 'nccl', 'init_method': 'env://'},
    'output': {
        'output_dir':   OUTPUT_DIR,
        'save_postfix': 'Perovskite_E2a_sad',   # new postfix -> fresh run
        'log_file':     'training_perovskite_sad.log',
    },
    'seed':      42,
    'device':    'cuda',
    'test_run':  False,
    'eval_only': False,
}

RUN_CFG = f'{OUTPUT_DIR}/run_config.yaml'
with open(RUN_CFG, 'w') as fh:
    yaml.safe_dump(cfg, fh, sort_keys=False)
print('Config written to', RUN_CFG)
print('start_sad:', cfg['training']['start_sad'], '| arch:', cfg['model']['architecture'],
      '| core_weight:', cfg['training']['core_weight'])


Config written to /content/drive/MyDrive/chargeflow/output_perovskite/run_config.yaml
start_sad: True | arch: unet_cond_xlarge | core_weight: 1.0


## 4. Download + extract perovskite data (ONCE — lives on Drive)

Skip on later sessions — data is already on Drive if the marker file exists.

In [129]:
import os
ZENODO = 'https://zenodo.org/records/19211405/files'
files = [
    'perovskites_inputs_initial_density.tar.gz',
    'perovskites_targets_dft_density.tar.gz',
    'external_periodic_test_perovskites_split.csv',
    'paper_splits_all.csv',
]
marker = f'{DATA_DIR}/.extracted'
if not os.path.exists(marker):
    for f in files:
        local = f'/content/{f}'
        !wget -q -c "{ZENODO}/{f}" -O "{local}"
        if f.endswith('.tar.gz'):
            !tar -xzf "{local}" -C "{DATA_DIR}"
        else:
            !cp "{local}" "{DATA_DIR}/"
    open(marker, 'w').close()
    print('Extracted to', DATA_DIR)
else:
    print('Already extracted — skipping.')


## 5. Inspect extracted filenames

Run once to confirm folder/file structure before building splits.

In [130]:
import glob, os
npys = glob.glob(f'{DATA_DIR}/**/*.npy', recursive=True)
print('total .npy:', len(npys))
print('--- first 10 ---')
for p in sorted(npys)[:10]:
    print(p)
# show top-level subdirs
print('\n--- top-level subdirs ---')
for d in sorted(os.listdir(DATA_DIR)):
    full = f'{DATA_DIR}/{d}'
    if os.path.isdir(full):
        n = len(os.listdir(full))
        print(f'  {d}/  ({n} entries)')


## 6. Build parent-disjoint train/test split + dataset file lists

Based on actual filenames from Colab output:
```
perovskites_rho_initial/VASP_mp-1055491_Rb1Ag1F3_remove_23_charge_-1/rho_22.npy
```
- Inputs folder: `perovskites_rho_initial/`
- Targets folder: `perovskites_rho_dft/` (or similar — confirmed by the inspect cell above)
- Pairing key: subfolder name `VASP_mp-XXXXX_..._charge_N`
- Parent id: `mp-XXXXX` (first MP id in the subfolder name)

The gridsize is recovered from the `.npy` array shape (written as a sidecar `.txt`).

In [131]:
import re, glob, random, os
import numpy as np
random.seed(42)

# --- identify input and target root dirs ---
# 'perovskites_rho_initial' = SAD/neutral inputs ; 'perovskites_rho' = DFT targets
subdirs = [d for d in os.listdir(DATA_DIR)
           if os.path.isdir(f'{DATA_DIR}/{d}') and d.startswith('perovskites')]
print('perovskite subdirs:', subdirs)

INPUT_SUBDIR  = next((d for d in subdirs if 'initial' in d or 'input' in d), None)
TARGET_SUBDIR = next((d for d in subdirs if d != INPUT_SUBDIR), None)
assert INPUT_SUBDIR and TARGET_SUBDIR, f'Cannot resolve input/target in {subdirs}'
print('inputs :', INPUT_SUBDIR, '\ntargets:', TARGET_SUBDIR)

INPUT_ROOT  = f'{DATA_DIR}/{INPUT_SUBDIR}'
TARGET_ROOT = f'{DATA_DIR}/{TARGET_SUBDIR}'

# --- pair input/target by matching subfolder name ---
def find_npy(root_dir):
    result = {}
    for subdir in os.listdir(root_dir):
        p = f'{root_dir}/{subdir}'
        if os.path.isdir(p):
            npys = glob.glob(f'{p}/*.npy')
            if npys:
                result[subdir] = npys[0]
    return result

inp_map = find_npy(INPUT_ROOT)
tgt_map = find_npy(TARGET_ROOT)
common  = sorted(set(inp_map) & set(tgt_map))
pairs   = [(inp_map[k], tgt_map[k]) for k in common]
print(f'paired: {len(pairs)}')

def parent_id_of(subdir_name):
    m = re.search(r'(mp-\d+)', subdir_name)
    return m.group(1) if m else subdir_name

# --- gridsize sidecar from the REAL .npy shape (arrays are already 3D) ---
# The dataset does np.load(npy).reshape(1, *gridsize), so gridsize MUST equal
# the true array shape. We read it directly — no guessing.
def gridsize_path_for(npy_path):
    txt = npy_path.replace('.npy', '.realshape.txt')
    arr = np.load(npy_path, mmap_mode='r')
    assert arr.ndim == 3, f'Expected 3D array, got {arr.ndim}D for {npy_path}'
    with open(txt, 'w') as fh:
        fh.write(' '.join(map(str, arr.shape)))
    return txt

# --- parent-disjoint 85/15 split ---
parents  = sorted({parent_id_of(k) for k in common})
random.shuffle(parents)
n_test   = max(1, int(0.15 * len(parents)))
test_set = set(parents[:n_test])
train_pairs = [(i, t) for k, (i, t) in zip(common, pairs) if parent_id_of(k) not in test_set]
test_pairs  = [(i, t) for k, (i, t) in zip(common, pairs) if parent_id_of(k) in test_set]
print(f'parents: {len(parents)}  train: {len(train_pairs)}  test: {len(test_pairs)}')

def write_lists(split_pairs, tag):
    data_paths  = [p[0] for p in split_pairs]
    label_paths = [p[1] for p in split_pairs]
    data_gs     = [gridsize_path_for(p) for p in data_paths]
    label_gs    = [gridsize_path_for(p) for p in label_paths]
    for name, items in [
        (f'list_d_{tag}',   data_paths),
        (f'list_l_{tag}',   label_paths),
        (f'list_dgs_{tag}', data_gs),
        (f'list_lgs_{tag}', label_gs),
    ]:
        with open(f'{LISTS_DIR}/{name}', 'w') as fh:
            fh.write('\n'.join(items) + '\n')
    print(f'wrote {tag} lists ({len(split_pairs)} samples)')

write_lists(train_pairs, 'perovskite_train')
write_lists(test_pairs,  'perovskite_test')

# sanity check: confirm a written gridsize matches the npy shape
chk_npy = data_paths_check = open(f'{LISTS_DIR}/list_d_perovskite_train').readline().strip()
chk_gs  = open(f'{LISTS_DIR}/list_dgs_perovskite_train').readline().strip()
print('\nSanity:', np.load(chk_npy, mmap_mode='r').shape, '==', open(chk_gs).read())


In [132]:
# DIAGNOSTIC: check the actual shape of the .npy density files and whether
# the original grid_sizes_*.dat files were shipped with the data.
import glob, os, numpy as np

sample_dir = glob.glob(f'{DATA_DIR}/perovskites_rho_initial/*/')[0]
print('Sample structure folder:', sample_dir)
print('Files in it:', os.listdir(sample_dir))
print()

npy = glob.glob(f'{sample_dir}/*.npy')[0]
arr = np.load(npy)
print(f'{os.path.basename(npy)}: shape={arr.shape}, ndim={arr.ndim}, size={arr.size}')
if arr.ndim == 1:
    cube = round(arr.size ** (1/3))
    print(f'  flat array. cube-root = {cube} (cube={cube**3==arr.size})')
    print('  -> need the real grid_sizes_*.dat to reshape correctly!')

# Check a few more structures to see if shapes/sizes differ across the family
print('\nSizes across 8 structures:')
for d in sorted(glob.glob(f'{DATA_DIR}/perovskites_rho_initial/*/'))[:8]:
    p = glob.glob(f'{d}/*.npy')[0]
    a = np.load(p, mmap_mode='r')
    print(f'  {os.path.basename(d.rstrip("/"))[:40]:40s} shape={a.shape} size={a.size}')


## 7. Find latest checkpoint (auto-resume)

Run this at the start of every session — it picks up where the last session left off.

In [6]:
# Pull latest code fixes (REPO_DIR is the nested path resolved in cell 2)
!git -C {REPO_DIR} pull

# Find latest checkpoint for the CURRENT run's postfix, for auto-resume.
import glob, os
POSTFIX = 'Perovskite_E2a_sad'   # must match cfg['output']['save_postfix'] in cell 3
rolling = f'{OUTPUT_DIR}/checkpoint-charged-residual-{POSTFIX}.pth'
RESUME  = rolling if os.path.exists(rolling) else None
if RESUME is None:
    ck = glob.glob(f'{OUTPUT_DIR}/checkpoint-*-{POSTFIX}.pth')
    if ck:
        RESUME = max(ck, key=os.path.getmtime)
print('Resume from:', RESUME or '(none — fresh start)')


Already up to date.
Resume from: /content/drive/MyDrive/chargeflow/output_perovskite/checkpoint-charged-residual-Perovskite_E2a_sad.pth


In [134]:
!git -C /content/chargeflow pull


## 8. Train (resumable)

Runs until `epochs` or until you stop the cell. Each disconnect is safe — the last Drive checkpoint is intact.

**After your first epoch completes**, note the epoch time printed in the log and adjust `save_frequency` in the config dict above (cell 3) so checkpoints land every ~15 min, then re-run cell 3 + this cell.

In [8]:
resume_arg = f'--resume "{RESUME}"' if RESUME else ''
!cd {REPO_DIR} && python scripts/train.py --config "{RUN_CFG}" {resume_arg}


## 9. Final evaluation — the editor-gate ε_MAE number

Run after training converges. Uses the parent-disjoint test split. Target: **< 0.01 (1%)**.

In [12]:
# Editor-gate evaluation (start_sad=True / SAD->DFT flow, the paper's formulation).
# ODE starts from the SAD input (x_0 = SAD), integrates to the DFT density.
# Output IS the DFT prediction directly (NO + SAD). Charge index -> label,
# concat_conditioning = SAD (Option B: model gets SAD both as flow start AND concat).
# ε_MAE = sum(|pred - target|) / sum(target), averaged over structures.
import sys, os
for m in [k for k in list(sys.modules) if k == 'src' or k.startswith('src.')]:
    del sys.modules[m]
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

import torch, numpy as np
from src.data.dataset import RhoDatasetCharge
from src.models.model_configs import instantiate_model
from flow_matching.solver.ode_solver import ODESolver
from flow_matching.utils import ModelWrapper
from src.training.edm_time_discretization import get_time_discretization

class TrainArgs:  # stub so torch.load can unpickle the checkpoint's args
    def __init__(self, *a, **k): pass

device = torch.device('cuda')
NFE = 100
START_SAD = True                  # paper's SAD->DFT formulation
ARCH    = 'unet_cond_xlarge'      # match what you trained
POSTFIX = 'Perovskite_E2a_sad'    # match cfg save_postfix for the start_sad run

ckpt_path = f'{OUTPUT_DIR}/best_model-{POSTFIX}.pth'
if not os.path.exists(ckpt_path):
    ckpt_path = f'{OUTPUT_DIR}/checkpoint-charged-residual-{POSTFIX}.pth'
model = instantiate_model(architechture=ARCH, is_discrete=False, use_ema=True)
ckpt = torch.load(ckpt_path, map_location=device, weights_only=False)
state = ckpt['model'] if 'model' in ckpt else ckpt
if list(state.keys())[0].startswith('module.'):
    state = {k.replace('module.', ''): v for k, v in state.items()}
model.load_state_dict(state, strict=True); model.to(device).eval()
print(f"Checkpoint: {os.path.basename(ckpt_path)} (epoch {ckpt.get('epoch','?')})  arch={ARCH}")

class ChargeCFGModel(ModelWrapper):
    def forward(self, x, t, charge, concat):
        t = torch.zeros(x.shape[0], device=x.device) + t
        with torch.autocast('cuda'), torch.no_grad():
            return self.model(x, t, extra={"label": charge,
                                           "concat_conditioning": concat}).to(torch.float32)
wrapped = ChargeCFGModel(model); wrapped.eval()
solver = ODESolver(velocity_model=wrapped)

test_ds = RhoDatasetCharge(
    data_list_path     = f'{LISTS_DIR}/list_d_perovskite_test',
    label_list_path    = f'{LISTS_DIR}/list_l_perovskite_test',
    data_gridsize_path = f'{LISTS_DIR}/list_dgs_perovskite_test',
    label_gridsize_path= f'{LISTS_DIR}/list_lgs_perovskite_test',
    data_augmentation  = False,
)
loader = torch.utils.data.DataLoader(test_ds, batch_size=1, shuffle=False)
print(f'Test structures: {len(test_ds)}')

eps_list = []
time_grid = get_time_discretization(nfes=NFE).to(device)
for i, (sad, target, charge) in enumerate(loader):
    sad = sad.to(device); target = target.to(device).float(); charge = charge.to(device)
    x_0 = sad if START_SAD else torch.randn_like(sad)*torch.sqrt(torch.tensor(1e-05, device=device))
    with torch.no_grad():
        out = solver.sample(time_grid=time_grid, x_init=x_0, method='dopri5',
                            atol=1e-5, rtol=1e-5, step_size=None,
                            charge=charge, concat=sad).float()
    pred = out if START_SAD else (sad + out)   # start_sad: output IS the DFT density
    eps = (pred - target).abs().sum() / target.sum()
    eps_list.append(eps.item())
    print(f'  [{i+1:2d}/{len(test_ds)}] ε_MAE = {eps.item()*100:.3f}%')

import statistics
mean_eps = statistics.mean(eps_list)
print('\n' + '='*60)
print(f'  Mean ε_MAE: {mean_eps*100:.3f}%   median: {statistics.median(eps_list)*100:.3f}%')
print(f'  frac < 1%: {sum(e<0.01 for e in eps_list)/len(eps_list)*100:.0f}%')
print(f'  Editor gate (<1%): {"PASS" if mean_eps < 0.01 else "not yet"}')
print('='*60)


Checkpoint: best_model-Perovskite_E2a_sad.pth (epoch 1874)  arch=unet_cond_xlarge
Test structures: 24
  [ 1/24] ε_MAE = 8.485%
  [ 2/24] ε_MAE = 8.710%
  [ 3/24] ε_MAE = 8.097%
  [ 4/24] ε_MAE = 8.128%
  [ 5/24] ε_MAE = 5.572%
  [ 6/24] ε_MAE = 5.368%
  [ 7/24] ε_MAE = 5.209%
  [ 8/24] ε_MAE = 5.229%
  [ 9/24] ε_MAE = 5.417%
  [10/24] ε_MAE = 5.256%
  [11/24] ε_MAE = 5.171%
  [12/24] ε_MAE = 5.024%
  [13/24] ε_MAE = 4.848%
  [14/24] ε_MAE = 4.985%
  [15/24] ε_MAE = 4.736%
  [16/24] ε_MAE = 4.584%
  [17/24] ε_MAE = 4.465%
  [18/24] ε_MAE = 4.371%
  [19/24] ε_MAE = 4.050%
  [20/24] ε_MAE = 4.241%
  [21/24] ε_MAE = 6.125%
  [22/24] ε_MAE = 6.517%
  [23/24] ε_MAE = 5.906%
  [24/24] ε_MAE = 6.165%

  Mean ε_MAE: 5.694%   median: 5.243%
  frac < 1%: 0%
  Editor gate (<1%): not yet


In [13]:
# OVERFIT CHECK: eval ε_MAE on a few TRAINING structures.
# Uses the SAME SAD->DFT flow as cell 23 (start_sad=True): x_0 = SAD, output IS the
# DFT density (no + SAD). Earlier version started from noise and added SAD back — that
# was the OLD noise->residual flow and gave a bogus ~25% train ε_MAE. Now apples-to-apples.
# train≈test -> not overfitting; loss objective doesn't yield low ε_MAE (need bigger model / near-core loss).
# train<<test -> overfitting (need more data / regularization).
import torch, statistics
from src.data.dataset import RhoDatasetCharge

train_ds = RhoDatasetCharge(
    data_list_path     = f'{LISTS_DIR}/list_d_perovskite_train',
    label_list_path    = f'{LISTS_DIR}/list_l_perovskite_train',
    data_gridsize_path = f'{LISTS_DIR}/list_dgs_perovskite_train',
    label_gridsize_path= f'{LISTS_DIR}/list_lgs_perovskite_train',
    data_augmentation  = False,
)
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=1, shuffle=False)

tg = get_time_discretization(nfes=NFE).to(device)
train_eps = []
for i, (sad, target, charge) in enumerate(train_loader):
    sad = sad.to(device); target = target.to(device).float(); charge = charge.to(device)
    x_0 = sad if START_SAD else torch.randn_like(sad) * torch.sqrt(torch.tensor(1e-05, device=device))
    with torch.no_grad():
        out = solver.sample(time_grid=tg, x_init=x_0, method='dopri5',
                            atol=1e-5, rtol=1e-5, step_size=None,
                            charge=charge, concat=sad).float()
    pred = out if START_SAD else (sad + out)   # start_sad: output IS the DFT density
    train_eps.append(((pred - target).abs().sum() / target.sum()).item())
    if i >= 9:   # first 10 training structures is enough
        break

print(f'TRAIN ε_MAE (first 10): mean {statistics.mean(train_eps)*100:.2f}%')
print(f'TEST  ε_MAE (all 24):   mean {mean_eps*100:.2f}%')
print()
if statistics.mean(train_eps) < 0.5 * mean_eps:
    print('=> OVERFITTING: train much better than test. Need more data / regularization.')
else:
    print('=> NOT overfitting: train ~= test. Loss objective does not yield low ε_MAE.')
    print('   Levers: near-core-weighted loss, bigger model (unet_cond_xlarge).')


TRAIN ε_MAE (first 10): mean 3.69%
TEST  ε_MAE (all 24):   mean 5.69%

=> NOT overfitting: train ~= test. Loss objective does not yield low ε_MAE.
   Levers: near-core-weighted loss, bigger model (unet_cond_xlarge).


In [14]:
# Was the loss still descending at epoch 285, or had it plateaued?
# Reads the training log to decide if "more epochs" is worthwhile.
import json, os
log_path = f'{OUTPUT_DIR}/log.txt'
if os.path.exists(log_path):
    rows = [json.loads(l) for l in open(log_path) if l.strip()]
    losses = [(r.get('epoch'), r.get('train_loss')) for r in rows if 'train_loss' in r]
    print(f'{len(losses)} logged epochs')
    for e, l in losses[:5]:   print(f'  epoch {e:3d}: {l:.5f}')
    print('  ...')
    for e, l in losses[-10:]: print(f'  epoch {e:3d}: {l:.5f}')
else:
    print('No log.txt found at', log_path)


2826 logged epochs
  epoch   0: 0.05755
  epoch   1: 0.05681
  epoch   2: 0.05569
  epoch   3: 0.05412
  epoch   4: 0.05209
  ...
  epoch 1900: 0.00567
  epoch 1901: 0.00570
  epoch 1902: 0.00565
  epoch 1903: 0.00572
  epoch 1904: 0.00568
  epoch 1905: 0.00569
  epoch 1906: 0.00569
  epoch 1907: 0.00569
  epoch 1908: 0.00568
  epoch 1909: 0.00567


In [15]:
# DIAGNOSTIC: disambiguate "model bad" vs "eval wrong".
# Uses the SAME SAD->DFT flow as cell 23 (start_sad=True): x_0 = SAD, output IS the
# predicted DFT density. The model "correction" we compare is (pred - sad).
# 1) SAD baseline: eps_MAE of the raw input vs target (no model). The model must beat this.
# 2) correction magnitude: is the model moving SAD toward DFT, or barely moving it?
# 3) compare to the TRUE correction (target - sad) magnitude.
import torch, statistics

base_eps, corr_pred_mag, corr_true_mag = [], [], []
time_grid_d = get_time_discretization(nfes=NFE).to(device)
for i, (sad, target, charge) in enumerate(loader):
    sad = sad.to(device); target = target.to(device).float(); charge = charge.to(device)
    # baseline: just SAD
    base_eps.append(((sad - target).abs().sum() / target.sum()).item())
    # what the model predicted (SAD->DFT flow: output IS the DFT density)
    x_0 = sad if START_SAD else torch.randn_like(sad) * torch.sqrt(torch.tensor(1e-05, device=device))
    with torch.no_grad():
        out = solver.sample(time_grid=time_grid_d, x_init=x_0, method='dopri5',
                            atol=1e-5, rtol=1e-5, step_size=None,
                            charge=charge, concat=sad).float()
    pred = out if START_SAD else (sad + out)
    corr_pred_mag.append((pred - sad).abs().mean().item())   # how far the model moved SAD
    corr_true_mag.append((target - sad).abs().mean().item()) # how far it SHOULD move
    if i >= 5:
        break

print(f'SAD-only baseline ε_MAE (first 6): mean {statistics.mean(base_eps)*100:.2f}%')
print(f'Model correction |mean abs| (pred-sad, first 6): {statistics.mean(corr_pred_mag):.3e}')
print(f'TRUE correction |mean abs| (target-sad): {statistics.mean(corr_true_mag):.3e}')
print()
print('Interpretation:')
print(' - If SAD baseline ~= model eps: model is not improving on the input.')
print(' - If pred correction << true correction: model under-moves SAD (weak/under-trained).')
print(' - If pred correction ~ true correction but eps high: it moves the wrong places.')


SAD-only baseline ε_MAE (first 6): mean 19.37%
Model correction |mean abs| (pred-sad, first 6): 1.128e-01
TRUE correction |mean abs| (target-sad): 1.274e-01

Interpretation:
 - If SAD baseline ~= model eps: model is not improving on the input.
 - If pred correction << true correction: model under-moves SAD (weak/under-trained).
 - If pred correction ~ true correction but eps high: it moves the wrong places.


In [16]:
# ROOT-CAUSE CHECK: are SAD input and DFT target on the same scale?
# And what is the BEST POSSIBLE ε_MAE if the model predicted the TRUE residual exactly?
import torch, numpy as np, statistics

ideal_eps, sad_sum, dft_sum = [], [], []
for i, (sad, target, charge) in enumerate(loader):
    sad = sad.float(); target = target.float()
    sad_sum.append(sad.sum().item()); dft_sum.append(target.sum().item())
    # if model predicted the EXACT residual, pred = sad + (target - sad) = target -> eps=0.
    # So a nonzero "ideal" here would reveal a metric/scale bug. Check identity:
    ideal_pred = sad + (target - sad)
    ideal_eps.append(((ideal_pred - target).abs().sum() / target.sum()).item())
    if i >= 7: break

print('SAD   total (electrons):', [f'{s:.1f}' for s in sad_sum])
print('DFT   total (electrons):', [f'{s:.1f}' for s in dft_sum])
print('ratio DFT/SAD          :', [f'{d/s:.3f}' for s,d in zip(sad_sum,dft_sum)])
print(f'\n"ideal" ε_MAE (pred=exact residual): {statistics.mean(ideal_eps)*100:.4f}%  (must be ~0)')
print('\nIf DFT/SAD ratio is far from 1.0 -> different normalization/electron count.')
print('If totals look like raw electron counts (tens-hundreds) -> scales comparable.')


SAD   total (electrons): ['551201.3', '549639.9', '554324.3', '555885.8', '349059.0', '347741.8', '351693.4', '353010.6']
DFT   total (electrons): ['551201.4', '549639.9', '554324.3', '555885.8', '349058.9', '347741.8', '351693.4', '353010.6']
ratio DFT/SAD          : ['1.000', '1.000', '1.000', '1.000', '1.000', '1.000', '1.000', '1.000']

"ideal" ε_MAE (pred=exact residual): 0.0000%  (must be ~0)

If DFT/SAD ratio is far from 1.0 -> different normalization/electron count.
If totals look like raw electron counts (tens-hundreds) -> scales comparable.
